<a href="https://colab.research.google.com/github/MahammadWahab77/MockAIvoice/blob/main/Sf_Invoicing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gspread google-api-python-client

from google.colab import auth
auth.authenticate_user()

import google.auth
import gspread
from googleapiclient.discovery import build

# Auth
creds, _ = google.auth.default()
gc = gspread.authorize(creds)
drive = build("drive", "v3", credentials=creds)
docs  = build("docs", "v1", credentials=creds)

# IDs
SHEET_ID = "1JdZQBaphBvx44lkkxHBEtQR1q_QCrltJds8dM39sR2s"
TEMPLATE_FILE_ID = "139xtA8H-GHu0-U1Dk3wsIGae37sCVcD59rd5q_R6i6s"
DOCS_FOLDER_ID = "1FOPc7WSnRsd00uQ1EBJ7CmYXWH0iGOcz"

# Open sheet
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
headers = ws.row_values(1)

def col_num(name: str) -> int:
    return headers.index(name) + 1

required_cols = [
    "Status", "Invoice Number", "Bill To Name", "Ship To Name", "State",
    "Loan Applicant Name", "Place Of Supply", "Invoice Date", "Item Name",
    "Rate", "Taxable Amount", "Loan Amount", "Amount In Words", "Loan Tenure",
    "Doc ID", "Doc URL"
]
missing = [c for c in required_cols if c not in headers]
if missing:
    raise ValueError(f"Missing required columns in Sheet header: {missing}")

# Sheet column -> Template placeholder
mapping = {
    "Invoice Number": "FFInvoiceNumber",
    "Bill To Name": "FFBillToName",
    "Ship To Name": "FFShipToName",
    "State": "FFState",
    "Loan Applicant Name": "FFLoanApplicantName",
    "Place Of Supply": "FFPlaceOfSupply",
    "Invoice Date": "FFInvoiceDate",
    "Item Name": "FFItemName",
    "Rate": "FFRate",
    "Taxable Amount": "FFTaxableAmount",
    "Loan Amount": "FFLoanAmount",
    "Amount In Words": "FFTotalInWords",
    "Loan Tenure": "FFLoanTenure",
}

# Read all rows (as list of dicts)
records = ws.get_all_records()  # starts from row 2 as data

processed = 0
skipped = 0
errors = 0

for sheet_row_index, row in enumerate(records, start=2):
    status = str(row.get("Status", "")).strip()

    if status != "Generate Doc":
        skipped += 1
        continue

    try:
        invoice_no = str(row.get("Invoice Number", "")).strip()
        bill_to = str(row.get("Bill To Name", "")).strip()
        doc_name = f"{invoice_no}_{bill_to}".strip("_") or f"Invoice_Row_{sheet_row_index}"

        # 1) Copy template into Generated Docs folder
        copied = drive.files().copy(
            fileId=TEMPLATE_FILE_ID,
            body={"name": doc_name, "parents": [DOCS_FOLDER_ID]},
            fields="id"
        ).execute()

        doc_id = copied["id"]
        doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"

        # 2) Replace placeholders
        requests = []
        for sheet_col, placeholder in mapping.items():
            val = row.get(sheet_col, "")
            val_str = "" if val is None else str(val)
            requests.append({
                "replaceAllText": {
                    "containsText": {"text": placeholder, "matchCase": True},
                    "replaceText": val_str
                }
            })

        docs.documents().batchUpdate(documentId=doc_id, body={"requests": requests}).execute()

        # 3) Write back to sheet
        ws.update_cell(sheet_row_index, col_num("Doc ID"), doc_id)
        ws.update_cell(sheet_row_index, col_num("Doc URL"), doc_url)
        ws.update_cell(sheet_row_index, col_num("Status"), "Doc Generated")

        processed += 1
        print(f"[OK] Row {sheet_row_index}: {doc_url}")

    except Exception as e:
        errors += 1
        ws.update_cell(sheet_row_index, col_num("Status"), f"Error: {str(e)[:120]}")
        print(f"[ERR] Row {sheet_row_index}: {e}")

print("\nSummary:")
print("Processed:", processed)
print("Skipped:", skipped)
print("Errors:", errors)


[OK] Row 2: https://docs.google.com/document/d/1D_y8c0HDsesgDqbbugN9IRGZfIHEJjHt_1iRCvxcyqs/edit
[OK] Row 3: https://docs.google.com/document/d/12VshUA5adLxB7ZeWZ6DLJ9Nolt52L8kmRJb3NjG2vjQ/edit
[OK] Row 4: https://docs.google.com/document/d/1_lWw8tQR8sgNg3UyjPqZdykfWOA6A2pbx2aVOQW2OPI/edit
[OK] Row 5: https://docs.google.com/document/d/1SpYFiWgSCs5_ycGzMwMAoL0p0n2BYUb0QdQdUIMG2n0/edit
[OK] Row 6: https://docs.google.com/document/d/1T2Zb3qd3MVWRWaJF2otKMR_3ekKqYEIJa6bIC_9jNi4/edit

Summary:
Processed: 5
Skipped: 0
Errors: 0


In [ ]:
# =========================
# DOC GENERATION
# Only for (Status=Generate Doc AND verified status=Verified)
# =========================

# Define missing variables
QUEUE_STATUS = "Generate Doc"
DONE_STATUS = "Doc Generated"

# c1 should be the column mapping for the SF worksheet (sf_col from previous cell)
c1 = sf_col

# Placeholder log_event function if not defined elsewhere
def log_event(uid, nbfc_raw, category, status, message, action, url):
    # This is a placeholder. In a real scenario, this would write to a log file, a database, or Google Sheets.
    print(f"[LOG] UID: {uid}, NBFC: {nbfc_raw}, Category: {category}, Status: {status}, Message: {message}, Action: {action}, URL: {url}")

mapping = {
    "Invoice Number": "FFInvoiceNumber",
    "Bill To Name": "FFBillToName",
    "Ship To Name": "FFShipToName",
    "State": "FFState",
    "Loan Applicant Name": "FFLoanApplicantName",
    "Place Of Supply": "FFPlaceOfSupply",
    "Invoice Date": "FFInvoiceDate",
    "Item Name": "FFItemName",
    "Rate": "FFRate",
    "Taxable Amount": "FFTaxableAmount",
    "Loan Amount": "FFLoanAmount",
    "Amount In Words": "FFTotalInWords",
    "Loan Tenure": "FFLoanTenure",
}

# Re-read fresh rows after validation
rows1 = sf_ws.get_all_records() # Changed ws1 to sf_ws

doc_updates = []
doc_processed = doc_skipped = doc_errors = 0

# Optional: summarize skip reasons in console at end
skip_reason_counts = {}

def count_skip(reason):
    skip_reason_counts[reason] = skip_reason_counts.get(reason, 0) + 1

for r_idx, r in enumerate(rows1, start=2):
    uid = str(r.get("UID", "")).strip()
    nbfc_raw = str(r.get("NBFC Name", "")).strip()

    status = str(r.get("Status", "")).strip()
    vstat  = str(r.get("verified status", "")).strip()
    existing_doc = str(r.get("Doc ID", "")).strip()
    existing_url = str(r.get("Doc URL", "")).strip()

    # -------- Filter: Why is it skipping? (log every skip) --------
    # 1) Not in queue status
    if status != QUEUE_STATUS:
        reason = f"Skip: Status not '{QUEUE_STATUS}' (Found='{status}')"
        log_event(uid, nbfc_raw, "Invoice/Filter", "Skipped", reason, "No action", existing_url)
        count_skip(reason)
        doc_skipped += 1
        continue

    # 2) Not verified
    if vstat != "Verified":
        reason = f"Skip: verified status not 'Verified' (Found='{vstat}')"
        log_event(uid, nbfc_raw, "Invoice/Filter", "Skipped", reason, "No action", existing_url)
        count_skip(reason)
        doc_skipped += 1
        continue

    # 3) Missing UID (rare but possible)
    if not uid:
        reason = "Skip: Missing UID in Sheet1"
        log_event(uid, nbfc_raw, "Invoice/Filter", "Skipped", reason, "No action", existing_url)
        count_skip(reason)
        doc_skipped += 1
        continue

    # -------- Dedup: prevent duplicates --------
    if existing_doc:
        reason = f"Skip: Doc already exists (Doc ID={existing_doc})"
        log_event(uid, nbfc_raw, "Invoice/Dedup", "Skipped", reason, "No action", existing_url)
        count_skip(reason)
        doc_skipped += 1
        continue

    # -------- Create Invoice Doc --------
    try:
        invoice_no = str(r.get("Invoice Number", "")).strip()
        bill_to = str(r.get("Bill To Name", "")).strip()

        # Extra validation for visibility (so missing invoice/bill_to doesn't create weird names)
        if not invoice_no and not bill_to:
            reason = "Skip: Missing both Invoice Number and Bill To Name (cannot build doc name)"
            log_event(uid, nbfc_raw, "Invoice/Filter", "Skipped", reason, "No action", "")
            count_skip(reason)
            doc_skipped += 1
            continue

        doc_name = f"{invoice_no}_{bill_to}".strip("_") or f"Invoice_Row_{r_idx}"

        # Copy template
        copied = drive.files().copy(
            fileId=TEMPLATE_FILE_ID,
            body={"name": doc_name, "parents": [DOCS_FOLDER_ID]},
            fields="id"
        ).execute()

        doc_id = copied["id"]
        doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"

        # Replace placeholders
        reqs = []
        for sheet_col, placeholder in mapping.items():
            val = r.get(sheet_col, "")
            val_str = "" if val is None else str(val)
            reqs.append({
                "replaceAllText": {
                    "containsText": {"text": placeholder, "matchCase": False},
                    "replaceText": val_str
                }
            })

        docs.documents().batchUpdate(documentId=doc_id, body={"requests": reqs}).execute()

        # Write back to sheet
        doc_updates.append({"range": a1(r_idx, c1["Doc ID"]), "values": [[doc_id]]})
        doc_updates.append({"range": a1(r_idx, c1["Doc URL"]), "values": [[doc_url]]})
        doc_updates.append({"range": a1(r_idx, c1["Status"]), "values": [[DONE_STATUS]]})

        doc_processed += 1
        log_event(uid, nbfc_raw, "Invoice/Create", "Success", "Doc Generated", "Doc URL saved", doc_url)
        print(f"[DOC OK] Row {r_idx}: {doc_url}")

    except Exception as e:
        doc_errors += 1
        err_full = str(e)
        err_short = f"Error: {err_full[:500]}"  # log more detail
        # Write a short status to sheet but keep full info in logs
        doc_updates.append({"range": a1(r_idx, c1["Status"]), "values": [[f"Doc Error: {err_full[:120]}"]]})
        log_event(uid, nbfc_raw, "Invoice/Create", "Failed", err_short, "Invoice Creation Failed", "")
        print(f"[DOC ERR] Row {r_idx}: {err_full}")

# Apply doc updates
if doc_updates:
    sf_ws.batch_update(doc_updates) # Changed ws1 to sf_ws

print("\nDoc Generation Summary:")
print("Processed:", doc_processed)
print("Skipped:", doc_skipped)
print("Errors:", doc_errors)

if skip_reason_counts:
    print("\nSkip reason breakdown:")
    for k, v in sorted(skip_reason_counts.items(), key=lambda x: -x[1]):
        print(f"- {v}x {k}")


NameError: name 'sf_col' is not defined

In [ ]:
# =========================
# 1-CELL COLAB SCRIPT
# Salesforce (OAuth client_credentials) -> Google Sheet
# Uses getpass prompt for SF_CLIENT_SECRET (so it's not hardcoded)
# =========================

!pip -q install gspread gspread-dataframe pandas numpy num2words requests

import os, glob
import pandas as pd
import numpy as np
import requests
from num2words import num2words
import gspread
from gspread_dataframe import set_with_dataframe
import getpass

# -------------------------
# CONFIG
# -------------------------
SPREADSHEET_ID = "1JdZQBaphBvx44lkkxHBEtQR1q_QCrltJds8dM39sR2s"
TARGET_SHEET_NAME = "Sheet1"

# If this exact path doesn't exist, the code will auto-pick the matching json in /content
SERVICE_ACCOUNT_FILE = "/content/gen-lang-client-0280406724-1b915cfedc51.json"

SF_LOGIN_DOMAIN = "https://computing-ability-6555.my.salesforce.com"
SF_API_VERSION = "61.0"

SF_CLIENT_ID = "3MVG9Ijq7vc89psqxFx7Cb6LjE35hIXcR_f9apHONFxu9uCspcJhFK5zrvz5ZGDBcug14_Nl3ndkZtwcucWXs"
SF_GRANT_TYPE = "client_credentials"

# -------------------------
# Resolve service account file path
# -------------------------
if not os.path.exists(SERVICE_ACCOUNT_FILE):
    candidates = sorted(glob.glob("/content/gen-lang-client-0280406724-1b915cfedc51*.json"))
    if not candidates:
        raise FileNotFoundError(
            f"Service account JSON not found.\n"
            f"Tried: {SERVICE_ACCOUNT_FILE}\n"
            f"Also searched: /content/gen-lang-client-0280406724-1b915cfedc51*.json"
        )
    SERVICE_ACCOUNT_FILE = candidates[0]

print("✅ Using service account file:", SERVICE_ACCOUNT_FILE)

# -------------------------
# Securely prompt SF secret (masked)
# -------------------------
SF_CLIENT_SECRET = os.environ.get("SF_CLIENT_SECRET", "").strip()
if not SF_CLIENT_SECRET:
    SF_CLIENT_SECRET = getpass.getpass("Enter SF_CLIENT_SECRET (input hidden): ").strip()
    os.environ["SF_CLIENT_SECRET"] = SF_CLIENT_SECRET  # keep for rest of session

if not SF_CLIENT_SECRET:
    raise ValueError("SF_CLIENT_SECRET still empty. Paste it when prompted.")

# -------------------------
# GOOGLE SHEETS AUTH
# -------------------------
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sh = gc.open_by_key(SPREADSHEET_ID)
try:
    ws = sh.worksheet(TARGET_SHEET_NAME)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=TARGET_SHEET_NAME, rows="1000", cols="30")

# -------------------------
# SALESFORCE AUTH (client_credentials)
# -------------------------
def get_sf_token():
    token_urls = [
        f"{SF_LOGIN_DOMAIN}/services/oauth2/token",
        "https://login.salesforce.com/services/oauth2/token",
        "https://test.salesforce.com/services/oauth2/token",
    ]
    data = {
        "grant_type": SF_GRANT_TYPE,
        "client_id": SF_CLIENT_ID,
        "client_secret": SF_CLIENT_SECRET,
    }
    last_err = None
    for url in token_urls:
        try:
            r = requests.post(url, data=data, timeout=30)
            if r.status_code == 200:
                j = r.json()
                return j["access_token"], j.get("instance_url", SF_LOGIN_DOMAIN)
            last_err = f"{url} -> {r.status_code}: {r.text[:500]}"
        except Exception as e:
            last_err = f"{url} -> {repr(e)}"
    raise RuntimeError(f"Failed to fetch Salesforce token. Last error: {last_err}")

access_token, instance_url = get_sf_token()
base = f"{instance_url}/services/data/v{SF_API_VERSION}"
print("✅ Salesforce token OK. Instance:", instance_url)

def sf_get(path, params=None):
    url = base + path
    headers = {"Authorization": f"Bearer {access_token}"}
    r = requests.get(url, headers=headers, params=params, timeout=60)
    if r.status_code >= 300:
        raise RuntimeError(f"SF GET failed {r.status_code}: {r.text[:1000]}")
    return r.json()

def sf_query_all(soql):
    out = []
    res = sf_get("/query", params={"q": soql})
    out.extend(res.get("records", []))
    while not res.get("done", True):
        next_url = res["nextRecordsUrl"]
        r = requests.get(instance_url + next_url, headers={"Authorization": f"Bearer {access_token}"}, timeout=60)
        if r.status_code >= 300:
            raise RuntimeError(f"SF pagination failed {r.status_code}: {r.text[:1000]}")
        res = r.json()
        out.extend(res.get("records", []))
    for rr in out:
        rr.pop("attributes", None)
    return out

# -------------------------
# YOUR FIELD MAPPING (as provided)
# -------------------------
# NBFC_Onboarding__c filter:
#   other_NBFC_PRE__c = 'EMI setup Done'
#
# NBFC fields:
#   NBFC Name        : Choose_NBFC__c (fallback Name)
#   Loan Tenure      : Master_Approved_Tenure__c
#   Taxable Amount   : Master_Approved_Loan_Amount__c
#   Loan Amount      : Master_Approved_Loan_Amount__c
#   Loan Applicant   : Co_Applicant_Name_PRE__c
#   Academy lookup   : Academy_Onboarding_PRE_L__c
#
# Academy_Onboarding_PRE__c fields:
#   UID              : userId__c
#   State            : State__c
#   Name             : Name (used for Bill To / Ship To; if you truly want the Id, replace with academy_id)

OUT_COLS = [
    "UID","NBFC Name","verified status","Loan Tenure","Taxable Amount","Loan Amount","Invoice Number",
    "Bill To Name","Ship To Name","State","Loan Applicant Name","Place Of Supply","Invoice Date",
    "Item Name","Rate","Amount In Words","Status","Doc ID","Doc URL","Validation Remarks","PDF URL","PDF ID"
]

STATE_TO_GST_CODE = {
    "Andhra Pradesh":"37","Arunachal Pradesh":"12","Assam":"18","Bihar":"10","Chhattisgarh":"22","Goa":"30","Gujarat":"24","Haryana":"06",
    "Himachal Pradesh":"02","Jharkhand":"20","Karnataka":"29","Kerala":"32","Madhya Pradesh":"23","Maharashtra":"27","Manipur":"14","Meghalaya":"17",
    "Mizoram":"15","Nagaland":"13","Odisha":"21","Punjab":"03","Rajasthan":"08","Sikkim":"11","Tamil Nadu":"33","Telangana":"36","Tripura":"16",
    "Uttar Pradesh":"09","Uttarakhand":"05","West Bengal":"19","Delhi":"07","Puducherry":"34","Jammu and Kashmir":"01","Ladakh":"38","Chandigarh":"04",
    "Andaman and Nicobar Islands":"35","Dadra and Nagar Haveli and Daman and Diu":"26",
}

def place_of_supply(state_name):
    if not state_name:
        return ""
    s = str(state_name).strip()
    code = STATE_TO_GST_CODE.get(s)
    return f"{s} ({code})" if code else s

def as_number(x):
    try:
        if x is None: return None
        if isinstance(x, float) and np.isnan(x): return None
        return float(x)
    except:
        return None

def inr_words(amount):
    amt = as_number(amount)
    if amt is None:
        return ""
    rupees = int(round(amt))
    return num2words(rupees, lang="en_IN").replace("-", " ").title() + " Rupees Only"

# -------------------------
# 1) Pull NBFC records
# -------------------------
# Detect whether Choose_NBFC__c exists; else fallback to Name
nbfc_desc = sf_get("/sobjects/NBFC_Onboarding__c/describe")
nbfc_fields = {f["name"] for f in nbfc_desc["fields"]}
nbfc_name_field = "Choose_NBFC__c" if "Choose_NBFC__c" in nbfc_fields else "Name"

NBFC_SELECT = [
    "Id",
    "Name",
    "other_NBFC_PRE__c",
    "Academy_Onboarding_PRE_L__c",
    "Master_Approved_Tenure__c",
    "Master_Approved_Loan_Amount__c",
    "Co_Applicant_Name_PRE__c",
]
if nbfc_name_field != "Name":
    NBFC_SELECT.append(nbfc_name_field)

soql_nbfc = f"""
SELECT {", ".join(NBFC_SELECT)}
FROM NBFC_Onboarding__c
WHERE other_NBFC_PRE__c = 'EMI setup Done'
"""

nbfc_rows = sf_query_all(soql_nbfc)
df_nbfc = pd.DataFrame(nbfc_rows)

if df_nbfc.empty:
    ws.clear()
    ws.update("A1", [["No records found for other_NBFC_PRE__c = 'EMI setup Done'"]])
    print("No NBFC records found. Sheet updated.")
    raise SystemExit()

# -------------------------
# 2) Pull Academy records for UID + State + Name
# -------------------------
acad_desc = sf_get("/sobjects/Academy_Onboarding_PRE__c/describe")
acad_fields = {f["name"] for f in acad_desc["fields"]}

required_acad = ["Id", "Name", "userId__c", "State__c"]
missing = [f for f in required_acad if f not in acad_fields]
if missing:
    raise RuntimeError(f"Academy_Onboarding_PRE__c missing fields: {missing}. Verify API names.")

academy_ids = df_nbfc["Academy_Onboarding_PRE_L__c"].dropna().astype(str).unique().tolist()

acad_map = {}
if academy_ids:
    def chunks(lst, n=200):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    acad_rows = []
    for ch in chunks(academy_ids, 200):
        ids_str = ",".join([f"'{x}'" for x in ch])
        soql_acad = f"""
        SELECT Id, Name, userId__c, State__c
        FROM Academy_Onboarding_PRE__c
        WHERE Id IN ({ids_str})
        """
        acad_rows.extend(sf_query_all(soql_acad))

    df_acad = pd.DataFrame(acad_rows)
    for _, rr in df_acad.iterrows():
        acad_map[str(rr["Id"])] = rr.to_dict()

# -------------------------
# 3) Build output
# -------------------------
out_rows = []
for _, r in df_nbfc.iterrows():
    academy_id = str(r.get("Academy_Onboarding_PRE_L__c") or "")
    acad = acad_map.get(academy_id, {}) if academy_id else {}

    uid = acad.get("userId__c", "")
    academy_name = acad.get("Name", "")
    state = acad.get("State__c", "")

    nbfc_name = r.get(nbfc_name_field, "") if nbfc_name_field in r else r.get("Name", "")
    tenure = r.get("Master_Approved_Tenure__c", "")
    loan_amt = r.get("Master_Approved_Loan_Amount__c", "")
    applicant_name = r.get("Co_Applicant_Name_PRE__c", "")

    # Bill/Ship To: you said Academy_Onboarding_PRE_L__c (Id).
    # In invoices, name is usually needed, so we use Academy Name; if you want Id, set bill_to=academy_id.
    bill_to = academy_name or academy_id
    ship_to = bill_to

    taxable = loan_amt
    rate = taxable

    remarks = []
    if not academy_id: remarks.append("Missing Academy_Onboarding_PRE_L__c")
    if not uid: remarks.append("Missing userId__c on Academy")
    if not state: remarks.append("Missing State__c on Academy")
    if as_number(loan_amt) is None: remarks.append("Missing/Invalid Master_Approved_Loan_Amount__c")
    if not tenure: remarks.append("Missing Master_Approved_Tenure__c")
    validation_remarks = "; ".join(remarks)

    out_rows.append({
        "UID": uid,
        "NBFC Name": nbfc_name,
        "verified status": "",
        "Loan Tenure": tenure,
        "Taxable Amount": taxable,
        "Loan Amount": loan_amt,
        "Invoice Number": "",
        "Bill To Name": bill_to,
        "Ship To Name": ship_to,
        "State": state,
        "Loan Applicant Name": applicant_name,
        "Place Of Supply": place_of_supply(state),
        "Invoice Date": "",
        "Item Name": "",
        "Rate": rate,
        "Amount In Words": inr_words(loan_amt),
        "Status": "EMI setup Done",
        "Doc ID": "",
        "Doc URL": "",
        "Validation Remarks": validation_remarks,
        "PDF URL": "",
        "PDF ID": "",
    })

df_out = pd.DataFrame(out_rows, columns=OUT_COLS)

# -------------------------
# 4) Write to Google Sheet
# -------------------------
ws.clear()
set_with_dataframe(ws, df_out, include_index=False, resize=True)

print(f"✅ Done. Pushed {len(df_out)} rows to Google Sheet -> {TARGET_SHEET_NAME}")
print("NBFC name field used:", nbfc_name_field)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 2.6 MB/s eta 0:00:00
✅ Using service account file: /content/gen-lang-client-0280406724-1b915cfedc51.json
Enter SF_CLIENT_SECRET (input hidden): ··········
✅ Salesforce token OK. Instance: https://computing-ability-6555.my.salesforce.com
✅ Done. Pushed 189 rows to Google Sheet -> Sheet1
NBFC name field used: Name


In [ ]:
# =========================
# 1-CELL COLAB SCRIPT (FIXED)
# Fixes:
# 1) Amount In Words -> now populated using num2words (en_IN)
# 2) Gyaandhan NBFC mapping improved -> Invoice Number will not be null
#    (Also: if NBFC still not recognized, we generate with code "X" but add a remark)
# =========================

!pip -q install gspread gspread-dataframe pandas numpy num2words requests

import os, glob
import pandas as pd
import numpy as np
import requests
import gspread
from gspread_dataframe import set_with_dataframe
import getpass
from datetime import datetime
from num2words import num2words

try:
    from zoneinfo import ZoneInfo
    IST = ZoneInfo("Asia/Kolkata")
except Exception:
    !pip -q install pytz
    import pytz
    IST = pytz.timezone("Asia/Kolkata")

# -------------------------
# CONFIG
# -------------------------
SPREADSHEET_ID = "1JdZQBaphBvx44lkkxHBEtQR1q_QCrltJds8dM39sR2s"
TARGET_SHEET_NAME = "Sheet1"
SERVICE_ACCOUNT_FILE = "/content/gen-lang-client-0280406724-1b915cfedc51.json"

SF_LOGIN_DOMAIN = "https://computing-ability-6555.my.salesforce.com"
SF_API_VERSION = "66.0"
SF_CLIENT_ID = "3MVG9Ijq7vc89psqxFx7Cb6LjE35hIXcR_f9apHONFxu9uCspcJhFK5zrvz5ZGDBcug14_Nl3ndkZtwcucWXs"
SF_GRANT_TYPE = "client_credentials"

# -------------------------
# Resolve service account file
# -------------------------
if not os.path.exists(SERVICE_ACCOUNT_FILE):
    candidates = sorted(glob.glob("/content/gen-lang-client-0280406724-1b915cfedc51*.json"))
    if not candidates:
        raise FileNotFoundError(
            f"Service account JSON not found.\nTried: {SERVICE_ACCOUNT_FILE}\n"
            f"Also searched: /content/gen-lang-client-0280406724-1b915cfedc51*.json"
        )
    SERVICE_ACCOUNT_FILE = candidates[0]
print("✅ Using service account file:", SERVICE_ACCOUNT_FILE)

# -------------------------
# Get Salesforce secret (hidden input)
# -------------------------
SF_CLIENT_SECRET = os.environ.get("SF_CLIENT_SECRET", "").strip()
if not SF_CLIENT_SECRET:
    SF_CLIENT_SECRET = getpass.getpass("Enter SF_CLIENT_SECRET (input hidden): ").strip()
    os.environ["SF_CLIENT_SECRET"] = SF_CLIENT_SECRET
if not SF_CLIENT_SECRET:
    raise ValueError("SF_CLIENT_SECRET is empty. Paste it when prompted.")

# -------------------------
# Google Sheets auth
# -------------------------
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sh = gc.open_by_key(SPREADSHEET_ID)
try:
    ws = sh.worksheet(TARGET_SHEET_NAME)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=TARGET_SHEET_NAME, rows="3000", cols="50")

# -------------------------
# Salesforce auth (client_credentials)
# -------------------------
def get_sf_token():
    token_urls = [
        f"{SF_LOGIN_DOMAIN}/services/oauth2/token",
        "https://login.salesforce.com/services/oauth2/token",
        "https://test.salesforce.com/services/oauth2/token",
    ]
    data = {
        "grant_type": SF_GRANT_TYPE,
        "client_id": SF_CLIENT_ID,
        "client_secret": SF_CLIENT_SECRET,
    }
    last_err = None
    for url in token_urls:
        try:
            r = requests.post(url, data=data, timeout=30)
            if r.status_code == 200:
                j = r.json()
                return j["access_token"], j.get("instance_url", SF_LOGIN_DOMAIN)
            last_err = f"{url} -> {r.status_code}: {r.text[:1200]}"
        except Exception as e:
            last_err = f"{url} -> {repr(e)}"
    raise RuntimeError(f"Failed to fetch Salesforce token. Last error: {last_err}")

access_token, instance_url = get_sf_token()
base = f"{instance_url}/services/data/v{SF_API_VERSION}"
print("✅ Salesforce token OK. Instance:", instance_url)

def sf_get(path, params=None):
    url = base + path
    headers = {"Authorization": f"Bearer {access_token}"}
    r = requests.get(url, headers=headers, params=params, timeout=60)
    if r.status_code >= 300:
        raise RuntimeError(f"SF GET failed {r.status_code}: {r.text[:2000]}")
    return r.json()

def sf_query_all(soql):
    out = []
    res = sf_get("/query", params={"q": soql})
    out.extend(res.get("records", []))
    while not res.get("done", True):
        next_url = res["nextRecordsUrl"]
        r = requests.get(instance_url + next_url, headers={"Authorization": f"Bearer {access_token}"}, timeout=60)
        if r.status_code >= 300:
            raise RuntimeError(f"SF pagination failed {r.status_code}: {r.text[:2000]}")
        res = r.json()
        out.extend(res.get("records", []))
    for rr in out:
        rr.pop("attributes", None)
    return out

def describe_fields(obj):
    desc = sf_get(f"/sobjects/{obj}/describe")
    return {f["name"] for f in desc["fields"]}

# -------------------------
# Helpers
# -------------------------
STATE_TO_GST_CODE = {
    "Andhra Pradesh":"37","Arunachal Pradesh":"12","Assam":"18","Bihar":"10","Chhattisgarh":"22","Goa":"30","Gujarat":"24","Haryana":"06",
    "Himachal Pradesh":"02","Jharkhand":"20","Karnataka":"29","Kerala":"32","Madhya Pradesh":"23","Maharashtra":"27","Manipur":"14","Meghalaya":"17",
    "Mizoram":"15","Nagaland":"13","Odisha":"21","Punjab":"03","Rajasthan":"08","Sikkim":"11","Tamil Nadu":"33","Telangana":"36","Tripura":"16",
    "Uttar Pradesh":"09","Uttarakhand":"05","West Bengal":"19","Delhi":"07","Puducherry":"34","Jammu and Kashmir":"01","Ladakh":"38","Chandigarh":"04",
    "Andaman and Nicobar Islands":"35","Dadra and Nagar Haveli and Daman and Diu":"26",
}

def place_of_supply(state_name):
    if not state_name:
        return ""
    s = str(state_name).strip()
    code = STATE_TO_GST_CODE.get(s)
    return f"{s} ({code})" if code else s

def as_number(x):
    try:
        if x is None: return None
        if isinstance(x, float) and np.isnan(x): return None
        return float(x)
    except:
        return None

def amount_in_words_rupees(amount):
    amt = as_number(amount)
    if amt is None:
        return ""
    rupees = int(round(amt))
    words = num2words(rupees, lang="en_IN").replace("-", " ").replace(",", "")
    # Title Case but keep it readable
    return f"{words.title()} Rupees Only"

def norm(s):
    return ("" if s is None else str(s)).strip().lower()

def detect_nbfc_code(nbfc_name):
    """
    Robust mapping:
      - Gyaandhan/GyanDhan/gyandhan/gyan dhan -> GP
      - Bajaj -> P
      - Northern Arc -> N
      - FeeMonk -> F
    """
    n = norm(nbfc_name)
    # Gyaandhan variations
    if ("gyandhan" in n) or ("gyaandhan" in n) or (("gyan" in n) and ("dhan" in n)) or (("gyaan" in n) and ("dhan" in n)):
        return "GP"
    if "bajaj" in n:
        return "P"
    if ("northern" in n) or ("northernarc" in n) or (("arc" in n) and ("northern" in n)):
        return "N"
    if "feemonk" in n or "fee monk" in n:
        return "F"
    return None

def item_name_from_program(program):
    p = norm(program)
    if "smart" in p:
        return "CCBP Academy Smart EMI"
    if "genius" in p:
        return "CCBP Academy Genius EMI"
    if "edge" in p:
        return "CCBP Academy Edge EMI"
    return None  # force validation (no generic fallback)

# Invoice date format: 11-Feb-26 (IST)
today_ist = datetime.now(IST).date()
invoice_date_str = today_ist.strftime("%d-%b-%y")  # 11-Feb-26

# FY code for invoice prefix (India FY Apr-Mar): 2526 for FY 2025-26
def fy_code_for_date(d):
    year = d.year
    if d.month >= 4:
        start = year
        end = year + 1
    else:
        start = year - 1
        end = year
    return f"{str(start)[-2:]}{str(end)[-2:]}"

FY_CODE = fy_code_for_date(today_ist)            # e.g., 2526
INVOICE_PREFIX = f"{FY_CODE}TG101"               # e.g., 2526TG101
PAD_BY_CODE = {"GP": 5, "P": 7, "N": 5, "F": 5, "X": 5}

def generate_invoice_number(nbfc_code, seq_num):
    code = nbfc_code if nbfc_code else "X"       # NEVER null
    pad = PAD_BY_CODE.get(code, 5)
    return f"{INVOICE_PREFIX}{code}{str(int(seq_num)).zfill(pad)}"

# -------------------------
# Field setup (your mapping)
# -------------------------
nbfc_fields = describe_fields("NBFC_Onboarding__c")
acad_fields = describe_fields("Academy_Onboarding_PRE__c")

NBFC_NAME_FIELD = "Choose_NBFC__c" if "Choose_NBFC__c" in nbfc_fields else "Name"

# Required NBFC fields
required_nbfc = [
    "Id","other_NBFC_PRE__c","Academy_Onboarding_PRE_L__c",
    "Master_Approved_Tenure__c","Master_Approved_Loan_Amount__c",
    "Invoice_status__c",
]
missing_nbfc = [f for f in required_nbfc if f not in nbfc_fields]
if missing_nbfc:
    raise RuntimeError(f"NBFC_Onboarding__c missing fields: {missing_nbfc}. Verify API names.")

# Required Academy fields
required_acad = ["Id","Name","userId__c","State__c","Program_PRE__c","Co_Applicant_Name__c"]
missing_acad = [f for f in required_acad if f not in acad_fields]
if missing_acad:
    raise RuntimeError(f"Academy_Onboarding_PRE__c missing fields: {missing_acad}. Verify API names.")

# -------------------------
# 1) Fetch NBFC records
# -------------------------
nbfc_select = [
    "Id",
    "Name",
    NBFC_NAME_FIELD if NBFC_NAME_FIELD != "Name" else None,
    "other_NBFC_PRE__c",
    "Invoice_status__c",
    "Academy_Onboarding_PRE_L__c",
    "Master_Approved_Tenure__c",
    "Master_Approved_Loan_Amount__c",
]
nbfc_select = [f for f in nbfc_select if f]

soql_nbfc = f"""
SELECT {", ".join(nbfc_select)}
FROM NBFC_Onboarding__c
WHERE other_NBFC_PRE__c = 'EMI setup Done'
"""

nbfc_rows = sf_query_all(soql_nbfc)
df_nbfc = pd.DataFrame(nbfc_rows)

if df_nbfc.empty:
    ws.clear()
    ws.update("A1", [[f"No records found for other_NBFC_PRE__c = 'EMI setup Done' on {invoice_date_str}"]])
    print("No NBFC records found. Sheet updated.")
    raise SystemExit()

# -------------------------
# 2) Fetch Academy records
# -------------------------
academy_ids = df_nbfc["Academy_Onboarding_PRE_L__c"].dropna().astype(str).unique().tolist()

acad_map = {}
if academy_ids:
    def chunks(lst, n=200):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    acad_rows = []
    for ch in chunks(academy_ids, 200):
        ids_str = ",".join([f"'{x}'" for x in ch])
        soql_acad = f"""
        SELECT Id, Name, userId__c, State__c, Program_PRE__c, Co_Applicant_Name__c
        FROM Academy_Onboarding_PRE__c
        WHERE Id IN ({ids_str})
        """
        acad_rows.extend(sf_query_all(soql_acad))

    df_acad = pd.DataFrame(acad_rows)
    for _, rr in df_acad.iterrows():
        acad_map[str(rr["Id"])] = rr.to_dict()

# -------------------------
# 3) Prepare invoice sequences per NBFC
# -------------------------
df_nbfc["_nbfc_name"] = df_nbfc.apply(
    lambda r: r.get(NBFC_NAME_FIELD, "") if NBFC_NAME_FIELD in r else r.get("Name", ""),
    axis=1
)

df_nbfc["_nbfc_code_raw"] = df_nbfc["_nbfc_name"].apply(detect_nbfc_code)

# IMPORTANT FIX: Fill unknown NBFC codes with "X" so invoice never becomes null
df_nbfc["_nbfc_code"] = df_nbfc["_nbfc_code_raw"].fillna("X")

# Stable ordering for sequence
df_nbfc = df_nbfc.sort_values(by=["_nbfc_code", "Id"], na_position="last").reset_index(drop=True)
df_nbfc["_seq"] = df_nbfc.groupby("_nbfc_code").cumcount() + 1

# -------------------------
# 4) Build output
# -------------------------
OUT_COLS = [
    "UID","NBFC Name","verified status","Loan Tenure","Taxable Amount","Loan Amount","Invoice Number",
    "Bill To Name","Ship To Name","State","Loan Applicant Name","Place Of Supply","Invoice Date",
    "Item Name","Rate","Amount In Words","Stage","Invoice Status","Status",
    "Doc ID","Doc URL","Validation Remarks","PDF URL","PDF ID"
]

out_rows = []
for _, r in df_nbfc.iterrows():
    academy_id = str(r.get("Academy_Onboarding_PRE_L__c") or "")
    acad = acad_map.get(academy_id, {}) if academy_id else {}

    uid = acad.get("userId__c", "")
    bill_ship_name = acad.get("Name", "")
    state = acad.get("State__c", "")
    program = acad.get("Program_PRE__c", "")
    loan_applicant = acad.get("Co_Applicant_Name__c", "")

    nbfc_name = r.get("_nbfc_name", "") or ""
    nbfc_code_raw = r.get("_nbfc_code_raw", None)   # None means unmapped
    nbfc_code = r.get("_nbfc_code", "X")
    seq = r.get("_seq", 1)

    tenure = r.get("Master_Approved_Tenure__c", "")
    loan_amt = r.get("Master_Approved_Loan_Amount__c", "")
    invoice_status = r.get("Invoice_status__c", "")
    stage = r.get("other_NBFC_PRE__c", "")

    # Item Name must be Smart/Genius/Edge (no generic)
    item_name = item_name_from_program(program)

    invoice_number = generate_invoice_number(nbfc_code, seq)

    taxable = loan_amt
    rate = taxable

    remarks = []
    if not nbfc_name: remarks.append("Missing NBFC Name")
    if nbfc_code_raw is None: remarks.append("NBFC not mapped (expected Gyaandhan/Bajaj/Northern Arc/FeeMonk) -> used code X")
    if not invoice_number: remarks.append("Invoice Number not generated (should not happen)")
    if not academy_id: remarks.append("Missing Academy_Onboarding_PRE_L__c on NBFC record")
    if not bill_ship_name: remarks.append("Missing Academy Name for Bill/Ship To")
    if not uid: remarks.append("Missing Academy userId__c")
    if not state: remarks.append("Missing Academy State__c")
    if not loan_applicant: remarks.append("Missing Academy Co_Applicant_Name__c")
    if not program: remarks.append("Missing Academy Program_PRE__c")
    if item_name is None: remarks.append("Program_PRE__c not Smart/Genius/Edge (Item Name invalid)")
    if as_number(loan_amt) is None: remarks.append("Missing/Invalid Master_Approved_Loan_Amount__c")
    if not tenure: remarks.append("Missing Master_Approved_Tenure__c")
    validation_remarks = "; ".join(remarks)

    out_rows.append({
        "UID": uid,
        "NBFC Name": nbfc_name,
        "verified status": "",
        "Loan Tenure": tenure,
        "Taxable Amount": taxable,
        "Loan Amount": loan_amt,
        "Invoice Number": invoice_number,
        "Bill To Name": bill_ship_name,
        "Ship To Name": bill_ship_name,
        "State": state,
        "Loan Applicant Name": loan_applicant,
        "Place Of Supply": place_of_supply(state),
        "Invoice Date": invoice_date_str,                       # 11-Feb-26
        "Item Name": item_name if item_name else "",            # blank if invalid, flagged
        "Rate": rate,
        "Amount In Words": amount_in_words_rupees(loan_amt),    # ✅ FIXED
        "Stage": stage,
        "Invoice Status": invoice_status,
        "Status": "Generate Docs",
        "Doc ID": "",
        "Doc URL": "",
        "Validation Remarks": validation_remarks,
        "PDF URL": "",
        "PDF ID": "",
    })

df_out = pd.DataFrame(out_rows, columns=OUT_COLS)

# -------------------------
# 5) Write to Google Sheet
# -------------------------
ws.clear()
set_with_dataframe(ws, df_out, include_index=False, resize=True)

print(f"✅ Done. Pushed {len(df_out)} rows to Google Sheet -> {TARGET_SHEET_NAME}")
print("Invoice prefix:", INVOICE_PREFIX, "| Invoice date:", invoice_date_str)
print("NBFC Name field used:", NBFC_NAME_FIELD)
print("Counts by NBFC code:", df_nbfc["_nbfc_code"].value_counts(dropna=False).to_dict())
print("Unmapped NBFC rows (code X):", int((df_nbfc["_nbfc_code"] == "X").sum()))
print("Rows with validation issues:", int((df_out["Validation Remarks"].str.len() > 0).sum()))


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.4 MB/s eta 0:00:00
✅ Using service account file: /content/gen-lang-client-0280406724-1b915cfedc51.json
Enter SF_CLIENT_SECRET (input hidden): ··········
✅ Salesforce token OK. Instance: https://computing-ability-6555.my.salesforce.com
✅ Done. Pushed 294 rows to Google Sheet -> Sheet1
Invoice prefix: 2627TG101 | Invoice date: 21-May-26
NBFC Name field used: Name
Counts by NBFC code: {'P': 160, 'F': 51, 'GP': 36, 'N': 24, 'X': 23}
Unmapped NBFC rows (code X): 23
Rows with validation issues: 23


In [ ]:
# ============================
# ONE-CELL COLAB: DOC + PDF GENERATOR
# ============================

!pip -q install gspread google-api-python-client

from google.colab import auth
auth.authenticate_user()

import google.auth, re, io
import gspread
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseUpload

# -----------------------------
# CONFIG (PASTE YOUR IDS HERE)
# -----------------------------
SHEET_ID = "1JdZQBaphBvx44lkkxHBEtQR1q_QCrltJds8dM39sR2s"
SHEET_TAB_NAME = "Sheet1"

TEMPLATE_FILE_ID = "1pvGURo7MtJSyqq3KkkNgk8TeG9oBxP6VZ-rUw3QIRXA"
DOCS_FOLDER_ID   = "1xLiHJvukOFpl9Q8dwXxsTMTWxjLvxTOg"

# IMPORTANT: create a Drive folder for PDFs and paste its id
PDFS_FOLDER_ID   = "1eLQXFyWOwt1Oik0UeUlTfPxMtarFwzKT"

QUEUE_STATUS = "Generate Doc"
FINAL_STATUS = "PDF Generated"
REQUIRE_VERIFIED = True

# If your org allows, set True to make PDFs accessible by anyone with the link
SET_PDF_ANYONE_READER = False

# -----------------------------
# AUTH + OPEN SHEET
# -----------------------------
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

drive = build("drive", "v3", credentials=creds)
docs  = build("docs",  "v1", credentials=creds)

sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet(SHEET_TAB_NAME)

# -----------------------------
# HELPERS
# -----------------------------
def s(x): return "" if x is None else str(x).strip()

def a1(r, c): return gspread.utils.rowcol_to_a1(r, c)

def sanitize_filename(name: str, max_len=140) -> str:
    name = re.sub(r"[\\/:*?\"<>|]+", "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    return (name[:max_len] if len(name) > max_len else name) or "Invoice"

def norm_nbfc(x: str) -> str:
    return " ".join((x or "").strip().lower().split())

def drive_doc_url(doc_id: str) -> str:
    return f"https://docs.google.com/document/d/{doc_id}/edit"

def drive_pdf_view_url(file_id: str) -> str:
    return f"https://drive.google.com/file/d/{file_id}/view"

# -----------------------------
# ENSURE COLUMNS EXIST (auto-add PDF URL if missing)
# -----------------------------
headers = ws.row_values(1)
col = {h: i+1 for i, h in enumerate(headers)}

NEEDED_COLS = [
    "UID","NBFC Name","verified status","Loan Tenure","Taxable Amount","Loan Amount",
    "Invoice Number","Bill To Name","Ship To Name","State","Loan Applicant Name",
    "Place Of Supply","Invoice Date","Item Name","Rate","Amount In Words",
    "Status","Doc ID","Doc URL","Validation Remarks"
]

missing = [c for c in NEEDED_COLS if c not in col]
if missing:
    raise ValueError(f"Missing required columns in header: {missing}")

# Add PDF URL column if not present
if "PDF URL" not in col:
    ws.add_cols(1)
    headers = ws.row_values(1)
    new_col_idx = len(headers) + 1  # after add_cols, header row still old length
    # Actually read row again after adding cols
    headers = ws.row_values(1)
    # If the new header cell is blank, set it
    if len(headers) < new_col_idx:
        # expand list (rare)
        pass
    ws.update_cell(1, len(headers)+1, "PDF URL")  # append at end
    headers = ws.row_values(1)
    col = {h: i+1 for i, h in enumerate(headers)}

# Add optional PDF ID column (nice for dedupe/debug)
if "PDF ID" not in col:
    ws.add_cols(1)
    ws.update_cell(1, len(ws.row_values(1))+1, "PDF ID")
    headers = ws.row_values(1)
    col = {h: i+1 for i, h in enumerate(headers)}

# -----------------------------
# TEMPLATE PLACEHOLDER MAP (edit tokens to match your template)
# -----------------------------
PLACEHOLDER_MAP = {
    "Invoice Number": "FFInvoiceNumber",
    "Bill To Name": "FFBillToName",
    "Ship To Name": "FFShipToName",
    "State": "FFState",
    "Loan Applicant Name": "FFLoanApplicantName",
    "Place Of Supply": "FFPlaceOfSupply",
    "Invoice Date": "FFInvoiceDate",
    "Item Name": "FFItemName",
    "Rate": "FFRate",
    "Taxable Amount": "FFTaxableAmount",
    "Loan Amount": "FFLoanAmount",
    "Amount In Words": "FFTotalInWords",
    "Loan Tenure": "FFLoanTenure",
    "UID": "FFUID",
    "NBFC Name": "FFNBFCName",
}

# -----------------------------
# NBFC BANK DETAILS (from your notes)
# -----------------------------
NBFC_BANK = {
    "fibe": {
        "account_name": "Nxtwave Disruptive Technologies Private Limited",
        "account_number": "NTWAV1000000000006",
        "ifsc": "ICIC0000106",
        "branch_name": "ICICI Bank Ltd."
    },
    "northern arc": {
        "account_name": "Nxtwave Disruptive Technologies Private Limited",
        "account_number": "NTWAV1000000000013",
        "ifsc": "ICIC0000106",
        "branch_name": "ICICI Bank Ltd."
    },
    "bajaj": {
        "account_name": "Nxtwave Disruptive Technologies Private Limited",
        "account_number": "NTWAV1000000000005",
        "ifsc": "ICIC0000106",
        "branch_name": "ICICI Bank Ltd."
    },
    "gyandhan": {
        "account_name": "Nxtwave Disruptive Technologies Private Limited",
        "account_number": "NTWAV1000000000035",
        "ifsc": "ICIC0000106",
        "branch_name": "ICICI Bank Ltd."
    },
}
DEFAULT_BANK = {
    "account_name": "Nxtwave Disruptive Technologies Private Limited",
    "account_number": "NTWAV1000000000008",
    "ifsc": "ICIC0000106",
    "branch_name": "ICICI Bank Ltd."
}
ALIASES = {
    "gyaandhan": "gyandhan",
    "gyandhan": "gyandhan",
    "gyan dhan": "gyandhan",
    "northernarc": "northern arc",
    "bajaj finserv": "bajaj",
}

def resolve_bank(nbfc_name: str):
    k = norm_nbfc(nbfc_name)
    # alias by compact key too
    compact = k.replace(" ", "")
    k2 = ALIASES.get(k, ALIASES.get(compact, k))
    return NBFC_BANK.get(k2, DEFAULT_BANK)

# These must exist in your template footer:
BANK_TOKENS = {
    "FFAccountName": None,
    "FFAccountNumber": None,
    "FFIFSC": None,
    "FFBranchName": None,
}

# -----------------------------
# REQUIRED FIELD VALIDATION
# -----------------------------
REQUIRED_FOR_INVOICE = [
    "UID","NBFC Name","verified status","Loan Tenure","Taxable Amount","Loan Amount",
    "Invoice Number","Bill To Name","Ship To Name","State","Loan Applicant Name",
    "Place Of Supply","Invoice Date","Item Name","Rate","Amount In Words"
]

BAD_EMPTY_MARKERS = {"#N/A", "#REF!", "NA", "N/A"}

def find_empty_fields(row: dict):
    missing = []
    for f in REQUIRED_FOR_INVOICE:
        v = row.get(f)
        vs = s(v)
        if (v is None) or (vs == "") or (vs in BAD_EMPTY_MARKERS):
            missing.append(f)
    return missing

# -----------------------------
# MAIN RUN
# -----------------------------
records = ws.get_all_records()

updates = []
processed = skipped = errors = 0

def set_cell(row_idx, col_name, value):
    updates.append({"range": a1(row_idx, col[col_name]), "values": [[value]]})

for row_idx, r in enumerate(records, start=2):
    status = s(r.get("Status"))
    vstat  = s(r.get("verified status"))
    doc_id_existing = s(r.get("Doc ID"))
    pdf_url_existing = s(r.get("PDF URL"))

    if status != QUEUE_STATUS:
        continue

    if REQUIRE_VERIFIED and vstat != "Verified":
        skipped += 1
        set_cell(row_idx, "Validation Remarks", f"Skipped: verified status='{vstat}'")
        continue

    # If already has PDF URL, skip
    if pdf_url_existing:
        skipped += 1
        set_cell(row_idx, "Validation Remarks", "Skipped: PDF already exists")
        continue

    # Validate empties
    empty_fields = find_empty_fields(r)
    if empty_fields:
        skipped += 1
        set_cell(row_idx, "Status", "Missing Data")
        set_cell(row_idx, "Validation Remarks", "Missing fields: " + ", ".join(empty_fields))
        continue

    try:
        invoice_no = s(r.get("Invoice Number"))
        bill_to = s(r.get("Bill To Name"))
        uid = s(r.get("UID"))
        nbfc = s(r.get("NBFC Name"))

        doc_name = sanitize_filename(f"{invoice_no}_{bill_to}_{uid}")

        # 1) Create / reuse doc
        if not doc_id_existing:
            copied = drive.files().copy(
                fileId=TEMPLATE_FILE_ID,
                body={"name": doc_name, "parents": [DOCS_FOLDER_ID]},
                fields="id"
            ).execute()
            doc_id = copied["id"]
            doc_url = drive_doc_url(doc_id)

            # 2) Placeholder replacements
            reqs = []

            # Map fields
            for sheet_col, token in PLACEHOLDER_MAP.items():
                reqs.append({
                    "replaceAllText": {
                        "containsText": {"text": token, "matchCase": True},
                        "replaceText": s(r.get(sheet_col))
                    }
                })

            # Bank tokens
            bank = resolve_bank(nbfc)
            bank_tokens = {
                "FFAccountName": bank["account_name"],
                "FFAccountNumber": bank["account_number"],
                "FFIFSC": bank["ifsc"],
                "FFBranchName": bank["branch_name"],
            }
            for token, val in bank_tokens.items():
                reqs.append({
                    "replaceAllText": {
                        "containsText": {"text": token, "matchCase": True},
                        "replaceText": val
                    }
                })

            docs.documents().batchUpdate(documentId=doc_id, body={"requests": reqs}).execute()

            # Write doc back
            set_cell(row_idx, "Doc ID", doc_id)
            set_cell(row_idx, "Doc URL", doc_url)
        else:
            doc_id = doc_id_existing
            doc_url = s(r.get("Doc URL")) or drive_doc_url(doc_id)
            set_cell(row_idx, "Doc URL", doc_url)

        # 3) Export to PDF and upload
        pdf_bytes = drive.files().export(fileId=doc_id, mimeType="application/pdf").execute()

        pdf_name = sanitize_filename(f"{invoice_no}_{bill_to}_{uid}") + ".pdf"
        media = MediaIoBaseUpload(io.BytesIO(pdf_bytes), mimetype="application/pdf", resumable=False)

        created = drive.files().create(
            body={"name": pdf_name, "parents": [PDFS_FOLDER_ID]},
            media_body=media,
            fields="id"
        ).execute()

        pdf_file_id = created["id"]

        if SET_PDF_ANYONE_READER:
            drive.permissions().create(
                fileId=pdf_file_id,
                body={"type": "anyone", "role": "reader"},
                fields="id"
            ).execute()

        pdf_url = drive_pdf_view_url(pdf_file_id)

        # Write PDF back
        set_cell(row_idx, "PDF ID", pdf_file_id)
        set_cell(row_idx, "PDF URL", pdf_url)
        set_cell(row_idx, "Status", FINAL_STATUS)
        set_cell(row_idx, "Validation Remarks", "Doc + PDF Generated")

        processed += 1
        print(f"[OK] Row {row_idx}: {pdf_url}")

    except Exception as e:
        errors += 1
        err = str(e)[:250]
        set_cell(row_idx, "Status", "Error")
        set_cell(row_idx, "Validation Remarks", f"Error: {err}")
        print(f"[ERR] Row {row_idx}: {e}")

# Batch update once
if updates:
    ws.batch_update(updates)

print("\nSUMMARY")
print("Processed:", processed)
print("Skipped:", skipped)
print("Errors:", errors)


[OK] Row 2: https://drive.google.com/file/d/1ac-Hif3zMN2N3SPUGLwfozLK8WBZvLSn/view
[OK] Row 3: https://drive.google.com/file/d/1PyvW7KbCfMosPLroifkbgkk2eUYUevlP/view

SUMMARY
Processed: 2
Skipped: 0
Errors: 0
